# 03 - Feature Engineering

Extract window-level PPG features and save them as a Delta table for ML training.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

feature_schema = T.StructType([
    T.StructField('mean_amplitude', T.DoubleType(), True),
    T.StructField('std_amplitude', T.DoubleType(), True),
    T.StructField('min_amplitude', T.DoubleType(), True),
    T.StructField('max_amplitude', T.DoubleType(), True),
    T.StructField('amplitude_range', T.DoubleType(), True),
    T.StructField('peak_count', T.IntegerType(), True),
    T.StructField('heart_rate_approx', T.DoubleType(), True),
    T.StructField('signal_energy', T.DoubleType(), True),
    T.StructField('mean_abs_change', T.DoubleType(), True),
    T.StructField('systolic_proxy', T.DoubleType(), True),
    T.StructField('diastolic_proxy', T.DoubleType(), True),
    T.StructField('pulse_pressure_proxy', T.DoubleType(), True),
])

@F.udf(feature_schema)
def extract_features(signal, sampling_rate):
    import numpy as np

    arr = np.asarray(signal or [], dtype=float)
    if arr.size == 0:
        arr = np.zeros(1, dtype=float)

    peaks = []
    minimum_distance = max(1, int(0.30 * int(sampling_rate or 100)))
    last_peak = -minimum_distance
    threshold = float(np.mean(arr) + 0.15 * np.std(arr))
    for index in range(1, arr.size - 1):
        if arr[index] > arr[index - 1] and arr[index] > arr[index + 1] and arr[index] >= threshold:
            if index - last_peak >= minimum_distance:
                peaks.append(index)
                last_peak = index

    duration_seconds = max(float(arr.size) / float(sampling_rate or 100), 1e-6)
    signal_min = float(np.min(arr))
    signal_max = float(np.max(arr))
    diff = np.diff(arr) if arr.size > 1 else np.array([0.0])
    systolic_proxy = float(np.percentile(arr, 95))
    diastolic_proxy = float(np.percentile(arr, 5))

    return (
        float(np.mean(arr)),
        float(np.std(arr)),
        signal_min,
        signal_max,
        float(signal_max - signal_min),
        int(len(peaks)),
        float(len(peaks) * 60.0 / duration_seconds),
        float(np.mean(np.square(arr))),
        float(np.mean(np.abs(diff))),
        systolic_proxy,
        diastolic_proxy,
        float(systolic_proxy - diastolic_proxy),
    )

clean_df = spark.table('cardio_ppg_clean')
feature_result_df = clean_df.withColumn('feature_result', extract_features(F.col('normalized_signal'), F.col('sampling_rate')))

features_df = feature_result_df.select(
    'patient_id', 'window_id', 'timestamp', 'sampling_rate',
    'cardiovascular_status', 'systolic_bp', 'diastolic_bp',
    F.col('feature_result.*')
)

features_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cardio_ppg_features')

print('Saved Delta table: cardio_ppg_features')
display(features_df.limit(10))
display(features_df.select('mean_amplitude', 'std_amplitude', 'peak_count', 'heart_rate_approx', 'signal_energy').summary())
